In [1]:
%load_ext autoreload
%autoreload 2

In [2]:


from src.main.python.iSel import cnn, enn, icf, lssm, lsbo, drop3, ldis, cdis, xldis, psdsp, ib3, cis, egdis, e2sc, biois, perplexity_is, autoencoder_is
from src.main.python.utils.general import get_data
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

/home/bernardo/projects/bio-is/venv/lib/python3.8/site-packages/sklearn/utils/deprecation.py:143: FutureWarning: The sklearn.neighbors.classification module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.neighbors. Anything that cannot be imported from sklearn.neighbors is now part of the private API.
  warnings.warn(message, FutureWarning)
Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib
/home/bernardo/projects/bio-is/venv/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

def get_selector(method: str):
    print(f"IS-Method: {method}")
    #Baselines
    if method == 'cnn':     return cnn.CNN()
    if method == 'enn':     return enn.ENN()
    if method == 'icf':     return icf.ICF()
    if method == 'lssm':    return lssm.LSSm()
    if method == 'lsbo':    return lsbo.LSBo()
    if method == 'drop3':   return drop3.DROP3()
    if method == 'ldis':    return ldis.LDIS()
    if method == 'cdis':    return cdis.CDIS()
    if method == 'xldis':   return xldis.XLDIS()
    if method == 'psdsp':   return psdsp.PSDSP()
    if method == 'ib3':     return ib3.IB3()
    if method == 'egdis':   return egdis.EGDIS()
    if method == 'cis':     return cis.CIS(task="atc")

    #proposed framework
    if method == 'e2sc-1':   return e2sc.E2SC(alphaMode="exact", betaMode='iterative')
    if method == 'e2sc-2':   return e2sc.E2SC(alphaMode="approximated", betaMode='heuristic')
    if method == 'bio-is':   return biois.BIOIS(beta=0.25, theta=0.50)

    # unsupervised methods
    if method == 'perplexity-is': return perplexity_is.PerplexityIS(n_topics=12, low_percentile=30, high_percentile=70, beta=0.3, theta=0.3)
    if method == 'autoencoder-is': return autoencoder_is.AutoencoderIS(
                                                                        n_epochs=10, batch_size=64, bottleneck_ratio=0.05, 
                                                                        beta=0.15, theta=0.15, low_percentile=25, high_percentile=75
                                                                    )
    return None


# Opening data - aisopos_ntua_2L dataset

In [4]:
inputdir = "/data/bernardolemos/datasets/webkb/tfidf/"

X_train, y_train, X_test, y_test, _ = get_data(inputdir, f=0)

# Example CNN - Selecting Instances

In [5]:
#selector = e2sc.E2SC(alphaMode="approximated", beta=0.15)
#selector = e2sc.E2SC(alphaMode="exact", betaMode='iterative')
#selector = get_selector(method="e2sc-1")
#selector = get_selector(method="e2sc-2")
# selector = get_selector(method="bio-is")
selector = get_selector(method="autoencoder-is")
# selector = get_selector(method="perplexity-is")
selector.fit(X_train, y_train)
idx = selector.sample_indices_
#print(idx)
X_train_selected, y_train_selected =  X_train[idx], y_train[idx]
selector.reduction_

IS-Method: autoencoder-is
[AE-IS] Using GPU: NVIDIA GeForce RTX 3090
[AE-IS] Architecture: 22983 → 20 → 10 → 20 → 22983
[AE-IS] Training autoencoder on 7376 instances (22983 features) for up to 10 epochs …
  Epoch   1/10 — loss: 1.120590 (best: 1.120590)
  Epoch  10/10 — loss: 1.002822 (best: 1.002822)
[AE-IS] Training finished — best loss: 1.002822
[AE-IS] Reconstruction error stats — min: 0.006810, median: 0.855894, max: 9.883924
[AE-IS] Redundancy threshold (p25): 0.499207 → 1843 candidates
[AE-IS] Noise threshold (p75): 1.321708 → 1844 candidates
[AE-IS] Removed 1566 redundant + 1567 noisy = 3133 total instances
[AE-IS] Kept 4243 / 7376 (reduction = 42.48%)


0.42475596529284165

# Example CNN - Comparing Classifiers

In [6]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"NoSel: {acc}")

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_selected, y_train_selected)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Selected: {acc}")


NoSel: 0.7812879708383961
Selected: 0.7715674362089915


# make_classification Example

In [7]:
from collections import Counter
from sklearn.datasets import make_classification
from src.main.python.iSel import e2sc
X, y = make_classification(n_classes=2, class_sep=2, weights=[0.1, 0.9], n_informative=3, n_redundant=1, flip_y=0, n_features=20, n_clusters_per_class=1, n_samples=1000, random_state=10)
print('Original dataset shape %s' % Counter(y))



Original dataset shape Counter({1: 900, 0: 100})


In [8]:
selector = get_selector(method="perplexity-is")
selector.fit(X, y)
idx = selector.sample_indices_
X_train_selected, y_train_selected =  X[idx], y[idx]
print('Resampled dataset shape %s' % Counter(y_train_selected))


IS-Method: perplexity-is
[PerplexityIS] Fitting LDA with 12 topics via 5-fold CV on 1000 instances …


/home/bernardo/projects/bio-is/src/main/python/iSel/perplexity_is.py:204: UserWarning: [PerplexityIS] Input contains negative values. Shifting features to non-negative range for LDA.
  warnings.warn(


[PerplexityIS] Perplexity stats — min: 1.0977, median: 2.1021, max: 4.2087
[PerplexityIS] Redundancy threshold (p30): 1.7344 → 300 candidates
[PerplexityIS] Noise threshold (p70): 2.1902 → 300 candidates
[PerplexityIS] Removed 210 redundant + 210 noisy = 420 total instances
[PerplexityIS] Kept 580 / 1000 (reduction = 42.00%)
Resampled dataset shape Counter({1: 547, 0: 33})


In [3]:
import pandas as pd
f = "/home/bernardo/projects/bio-is//data/bernardolemos/results/instance_selection/selection/webkb/split_10_autoencoder-is_idxinfold.pkl"
df_splits = pd.read_pickle(f)
df_splits

,train_idxs,test_idxs
0,"[1, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14, 18, 20, ...","[23, 29, 30, 32, 49, 51, 59, 65, 69, 70, 73, 7..."
1,"[2, 3, 4, 5, 7, 10, 11, 12, 13, 14, 16, 18, 20...","[15, 43, 44, 56, 63, 67, 78, 100, 101, 107, 11..."
2,"[1, 4, 5, 8, 9, 11, 12, 13, 14, 17, 19, 20, 21...","[18, 31, 54, 58, 81, 86, 111, 113, 174, 196, 2..."
3,"[1, 2, 3, 7, 8, 13, 14, 15, 16, 18, 20, 21, 24...","[10, 41, 48, 83, 88, 96, 126, 129, 131, 140, 1..."
4,"[0, 1, 2, 5, 6, 8, 10, 12, 14, 15, 16, 17, 18,...","[2, 3, 5, 6, 12, 24, 25, 27, 39, 45, 47, 55, 6..."
5,"[2, 3, 4, 6, 7, 8, 10, 11, 12, 14, 16, 17, 21,...","[0, 9, 33, 42, 52, 60, 62, 68, 71, 74, 77, 80,..."
6,"[1, 2, 3, 8, 9, 10, 11, 15, 16, 18, 20, 21, 22...","[7, 11, 28, 36, 38, 57, 61, 75, 79, 89, 90, 10..."
7,"[1, 2, 4, 5, 6, 7, 10, 12, 13, 14, 15, 16, 21,...","[4, 16, 17, 19, 22, 35, 46, 50, 93, 116, 119, ..."
8,"[1, 2, 3, 4, 6, 7, 11, 14, 15, 16, 17, 18, 19,...","[1, 8, 26, 37, 53, 103, 112, 122, 143, 146, 15..."
9,"[3, 5, 6, 7, 9, 11, 12, 13, 14, 15, 18, 19, 23...","[13, 14, 20, 21, 34, 40, 64, 87, 91, 95, 98, 1..."
